In [ ]:
!pip install ultralytics -q
import shutil
from pathlib import Path
import yaml
import torch
from torch.utils.data import Sampler
from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionTrainer
from ultralytics.data.build import build_yolo_dataset, build_dataloader
import cv2
import random
import numpy as np
from tqdm.notebook import tqdm
from torch.utils.data import DataLoader
print('Done.')

In [ ]:
# Replace with your Kaggle username and dataset name
SRC = Path('/kaggle/input/datasets/YOUR_USERNAME/YOUR_DATASET_NAME/processed_data')
DST = Path('/kaggle/working/processed_data')

print('Copying dataset to working directory...')
shutil.copytree(str(SRC), str(DST))
print('Done.')

In [ ]:
DATA_ROOT = Path('/kaggle/working/processed_data')

config = {
    'path' : str(DATA_ROOT),
    'train': 'images/train',
    'val' : 'images/val',
    'test' : 'images/test',
    'nc' : 4,
    'names': ['crazing', 'inclusion', 'patches', 'pitted_surface'],
}

yaml_path = Path('/kaggle/working/data.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

print(f'data.yaml saved to: {yaml_path}')
print(yaml.dump(config, default_flow_style=False, sort_keys=False))

In [ ]:
TRAIN_IMAGES = Path('/kaggle/working/processed_data/images/train')
TRAIN_LABELS = Path('/kaggle/working/processed_data/labels/train')
INCLUSION_CLASS_ID = 1
NUM_SYNTHETIC = 50
SEED = 42

random.seed(SEED)


def find_inclusion_tiles() -> list[Path]:
    inclusion_tiles = []
    for label_path in TRAIN_LABELS.glob('*.txt'):
        with open(label_path) as f:
            classes = [int(line.split()[0]) for line in f if line.strip()]
        if INCLUSION_CLASS_ID in classes:
            img_path = TRAIN_IMAGES / f'{label_path.stem}.jpg'
            if img_path.exists():
                inclusion_tiles.append(img_path)
    return inclusion_tiles

def flip_labels(label_path: Path, flipped: bool) -> list[str]:
    lines = []
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            if flipped:
                cls, x, y, w, h = parts
                x = str(round(1.0 - float(x), 6))
                parts = [cls, x, y, w, h]
            lines.append(' '.join(parts))
    return lines


def generate_synthetic_inclusion(inclusion_tiles: list[Path], n: int) -> None:
    sources = random.choices(inclusion_tiles, k=n)

    for idx, img_path in enumerate(tqdm(sources, desc='Generating synthetic inclusion', unit='tile')):
        label_path = TRAIN_LABELS / f'{img_path.stem}.txt'

        flipped = random.random() > 0.5
        image   = cv2.imread(str(img_path))

        if flipped:
            image = cv2.flip(image, 1)

        hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV).astype(np.float32)
        hsv[:, :, 0] *= random.uniform(0.95, 1.05)
        hsv[:, :, 1] *= random.uniform(0.80, 1.20)
        hsv[:, :, 2] *= random.uniform(0.80, 1.20)
        hsv = np.clip(hsv, 0, 255).astype(np.uint8)
        image = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

        alpha = random.uniform(0.85, 1.15)
        beta  = random.randint(-15, 15)
        image = np.clip(image * alpha + beta, 0, 255).astype(np.uint8)

        out_name  = f'synthetic_inclusion_{idx:04d}'
        out_img   = TRAIN_IMAGES / f'{out_name}.jpg'
        out_label = TRAIN_LABELS / f'{out_name}.txt'

        cv2.imwrite(str(out_img), image)

        lines = flip_labels(label_path, flipped)
        with open(out_label, 'w') as f:
            f.write('\n'.join(lines) + '\n')

print('Finding inclusion tiles...')
inclusion_tiles = find_inclusion_tiles()
print(f'Found {len(inclusion_tiles)} inclusion tiles in train set.')

generate_synthetic_inclusion(inclusion_tiles, NUM_SYNTHETIC)
print(f'Done. Added {NUM_SYNTHETIC} synthetic inclusion tiles to train set.')

In [ ]:
DEFECT_RATIO = 0.70
SEED = 42

def get_tile_lists() -> tuple[list[str], list[str]]:
    defect = []
    negative = []

    for label_path in TRAIN_LABELS.glob('*.txt'):
        with open(label_path) as f:
            content = f.read().strip()
        if content:
            defect.append(label_path.stem)
        else:
            negative.append(label_path.stem)

    return defect, negative


class BalancedSampler(Sampler):
    def __init__(
        self,
        dataset,
        defect_ratio: float = 0.70,
        seed: int = 42,
    ):
        self.defect_indices = []
        self.negative_indices = []

        for idx, label in enumerate(dataset.labels):
            if len(label['bboxes']) > 0:
                self.defect_indices.append(idx)
            else:
                self.negative_indices.append(idx)

        self.defect_ratio = defect_ratio
        self.seed = seed
        self.epoch = 0

        n_neg = int(len(self.defect_indices) * (1 - defect_ratio) / defect_ratio)
        self.n_neg = min(n_neg, len(self.negative_indices))

        print(f'Defect tiles:    {len(self.defect_indices)}')
        print(f'Negative tiles:  {len(self.negative_indices)}')
        print(f'Per-epoch total: {len(self.defect_indices) + self.n_neg} '
              f'({self.n_neg} negatives per epoch)')

    def set_epoch(self, epoch: int) -> None:
        self.epoch = epoch

    def __iter__(self):
        rng = random.Random(self.seed + self.epoch)
        negs = rng.sample(self.negative_indices, self.n_neg)
        indices = self.defect_indices + negs
        rng.shuffle(indices)
        return iter(indices)

    def __len__(self) -> int:
        return len(self.defect_indices) + self.n_neg


class BalancedDetectionTrainer(DetectionTrainer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

    def get_dataloader(self, dataset_path, batch_size, rank=0, mode='train'):
        dataset = build_yolo_dataset(
            self.args,
            dataset_path,
            batch_size,
            self.data,
            mode = mode,
            rect = False,
            stride = self.model.stride if hasattr(self.model, 'stride') else 32,
        )
    
        if mode == 'train':
            epoch = getattr(self, 'epoch', 0)
            if not hasattr(self, '_sampler'):
                self._sampler = BalancedSampler(dataset, DEFECT_RATIO, SEED)
            self._sampler.set_epoch(epoch)
            return DataLoader(
                dataset,
                batch_size = batch_size,
                sampler = self._sampler,
                num_workers = self.args.workers,
                pin_memory = True,
                collate_fn = getattr(dataset, 'collate_fn', None),
            )
    
        return build_dataloader(dataset, batch_size, self.args.workers,
                                shuffle=False, rank=rank)

# Replace with your weights path
model = YOLO('/kaggle/input/models/YOUR_USERNAME/YOUR_MODEL_NAME/pytorch/default/1/last.pt')

model.train(
    data = '/kaggle/working/data.yaml',
    epochs = 50,
    batch = 32,
    imgsz = 608,
    device = '0',
    workers = 4,
    project = '/kaggle/working/runs',
    name = 'severstal_yolov8m_balanced',
    patience = 20,
    resume = False,  # Set to True and update model path above to resume from checkpoint
    trainer = BalancedDetectionTrainer,
    
    hsv_h = 0.01,
    hsv_s = 0.3,
    hsv_v = 0.2,
    
    fliplr = 0.5,
    flipud = 0.0,
    mosaic = 0.5,
    mixup = 0.0,
    amp = True,
    close_mosaic = 17
)

In [ ]:
shutil.make_archive('/kaggle/working/weights_backup', 'zip', '/kaggle/working/runs/severstal_yolov8m_balanced/weights')
print('Done.')